# E1 + E2 · Error diagnosis and majority voting (no training)

Tasks **E1** and **E2** from `Essential_task_delegation/Next_Steps_Roadmap.md`. E1 was called P2-0 in Phase2_Plan.

**What it does**
- Runs the **R9** champion, its second seed **R9s7**, and the **R1** reference over the same 645 validation images as every D4 run. The E-protocol applies: full native 1024 images, standard and 8-view TTA.
- **E1:**
  - the 4×4 confusion matrices and building IoU;
  - where the missed destroyed pixels go. The share predicted as Background decides **Gate G2**: R11 or R12 first;
  - Destroyed IoU per disaster event and per source (xBD vs other).
- **E2:** object-based majority voting, from the BRIGHT paper. Every predicted building blob takes its majority class; it's scored with and without a 20 px minimum blob size.
- **Exports** R9's predicted masks (standard and TTA) and the ground-truth masks as PNGs. Tasks H1, T1 and H2 reuse them.

**How to run on Kaggle**
1. Settings: GPU T4 ×1, internet ON.
2. Inputs: the DisasterM3 mirror (`abrarmohammedtanzim/disasterm3-mirror`, attached automatically when you import this notebook).
3. Secret: `HF_TOKEN` (read access to the private `AbrarAlam/disasterm3-d4-*` repos). Checkpoints are downloaded from Hugging Face.
   - **Fallback:** if a repo has no `final_model.pth` (for example R9, which ran on another account), add a Kaggle dataset containing `R9/final_model.pth`, made from `Champion_Task/R9-results/d4_R9/`.
4. First run with `SMOKE_TEST = True` (8 images, a few minutes). Then set it to `False` and use Save & Run All (about 1 hour for three models).
5. Outputs in `/kaggle/working/E1/`:
   - `E1_summary.csv`, `E1_per_event.csv`, `E1_per_image_cm.csv`, `E1_confusion_matrices.json`;
   - `E2_voting.csv`, `config.json` and `README.md`;
   - the `preds/` and `gt/` folders.

   Turn the output into a Kaggle dataset named `d4-r9-val-predictions` for H1, T1 and H2.

**Done when:**
- the sanity check in Cell 7 shows every model matches its D4 CSV;
- the Gate G2 line is printed.

In [1]:
# ── Cell 1: Environment Setup (Run-All safe, identical to the D4 notebooks) ──
# Kaggle's image moved to Python 3.13, which has no torch 2.4.1 wheel. Use Kaggle's own
# torch/torchvision/opencv and pin only the libraries whose API this notebook depends on.
import importlib.metadata as _md
import os, sys, subprocess

PINS = {
    "segmentation-models-pytorch": "0.3.4",   # baseline version (checkpoint key layout)
    "albumentations": "1.4.21",               # 1.x API (PadIfNeeded value=/mask_value=)
}

def _mismatched():
    out = []
    for pkg, want in PINS.items():
        try:
            have = _md.version(pkg)
        except _md.PackageNotFoundError:
            have = "not installed"
        if have != want:
            out.append(f"{pkg}: {have} → {want}")
    return out

if _mismatched():
    print("⏳ Installing pinned versions:")
    for _m in _mismatched():
        print(f"   {_m}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *(f"{k}=={v}" for k, v in PINS.items()), "huggingface_hub[hf_xet]"],
                   check=True)                # pip failure stops the run here
    if _mismatched():
        raise RuntimeError(f"Pins still wrong after install: {_mismatched()}")
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
        # background "Save & Run All": nothing imported yet, so new versions load on first import
        print("✓ Dependencies installed (background run) — continuing without restart.")
    else:
        raise SystemExit("✓ Dependencies installed. RESTART THE KERNEL NOW "
                         "(Run → Restart & clear cell outputs), then Run All again.")
else:
    print("✓ All pinned versions already installed — proceeding.")

import torch
print(f"  torch {torch.__version__}, python {sys.version.split()[0]}")

⏳ Installing pinned versions:
   segmentation-models-pytorch: not installed → 0.3.4
   albumentations: 2.0.8 → 1.4.21
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.5/109.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.9/227.9 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 55.1 MB/s eta 0:00:00
✓ Dependencies installed (background run) — continuing without restart.
  torch 2.11.0+cu128, python 3.13.15


In [2]:
# ── Cell 2: Configuration ──────────────────────────────────────────────────
import glob
import json
import os
import re
import time
from pathlib import Path

import numpy as np
import torch
from huggingface_hub import hf_hub_download

DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"
OUT_DIR = Path("/kaggle/working/E1")

NUM_CLASSES = 4                     # 0=Background, 1=Intact, 2=Damaged, 3=Destroyed
CLASS_NAMES = ["Background", "Intact", "Damaged", "Destroyed"]
SMOKE_TEST = False                  # True = first 8 validation images only
RUN_MODELS = ["R9", "R1", "R9s7"]   # R9s7 lets E2 check that voting repeats on a second seed
EXPORT_PREDS_FOR = "R9"             # whose masks are saved for H1, T1 and H2
VOTE_MIN_AREA = 20                  # E2 variant: blobs smaller than this keep their pixel labels
HF_USER = "AbrarAlam"               # checkpoints: {HF_USER}/disasterm3-d4-<run>/final_model.pth

# Set a path here to skip the search, e.g. "/kaggle/input/my-ckpts/R9/final_model.pth"
CKPT_PATHS = {"R9": None, "R1": None, "R9s7": None}

# Each model must reproduce its D4 CSV (sanity check in Cell 7): (mIoU, Destroyed IoU)
EXPECTED = {
    "R9":   {"std": (0.5373, 0.2921), "tta": (0.5440, 0.3001)},
    "R1":   {"std": (0.5374, 0.2686), "tta": (0.5440, 0.2822)},
    "R9s7": {"std": (0.5382, 0.2847), "tta": (0.5453, 0.2941)},
}

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

def find_checkpoint(name):
    """Order: CKPT_PATHS, then a /kaggle/input folder named R9 / d4_R9 / R9-results, then Hugging Face."""
    if CKPT_PATHS.get(name):
        return CKPT_PATHS[name]
    wanted = {name.lower(), f"d4_{name}".lower(), f"{name}-results".lower()}
    hits = [p for p in glob.glob("/kaggle/input/**/*.pth", recursive=True)
            if any(part.lower() in wanted for part in Path(p).parts)]
    hits.sort(key=lambda p: Path(p).name != "final_model.pth")   # prefer final_model.pth
    if hits:
        return hits[0]
    repo = f"{HF_USER}/disasterm3-d4-{name}"
    errors = []
    for filename in ("final_model.pth", "last.pth"):   # last.pth holds the same epoch-40 weights
        try:
            return hf_hub_download(repo, filename, token=hf_token())
        except Exception as e:
            errors.append(f"{filename}: {type(e).__name__}")
    raise FileNotFoundError(
        f"No checkpoint for {name}: nothing under /kaggle/input and {repo} gave {errors}. "
        f"Add a Kaggle dataset with {name}/final_model.pth or set CKPT_PATHS['{name}'].")

MODELS = {name: find_checkpoint(name) for name in RUN_MODELS}
OUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"✓ E1 + E2 diagnosis{'  [SMOKE TEST]' if SMOKE_TEST else ''}")
for name, path in MODELS.items():
    print(f"  {name}: {path}")
print(f"  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"  Outputs: {OUT_DIR}")

final_model.pth: reconstructing file:   0%|          |  0.00B /  130MB            

final_model.pth: downloading bytes:           |  0.00B            

final_model.pth: reconstructing file:   0%|          |  0.00B /  130MB            

final_model.pth: downloading bytes:           |  0.00B            

final_model.pth: reconstructing file:   0%|          |  0.00B /  130MB            

final_model.pth: downloading bytes:           |  0.00B            

✓ E1 + E2 diagnosis
  R9: /root/.cache/huggingface/hub/models--AbrarAlam--disasterm3-d4-R9/snapshots/34897e1aa70742cac62a379561e1f969dccd4e10/final_model.pth
  R1: /root/.cache/huggingface/hub/models--AbrarAlam--disasterm3-d4-R1/snapshots/ebe008702466f2db4dfac0d1c51d269e2974f59a/final_model.pth
  R9s7: /root/.cache/huggingface/hub/models--AbrarAlam--disasterm3-d4-R9s7/snapshots/b97ed386d32c9b9fb9c2976c0fb70eae9462bb2a/final_model.pth
  Device: Tesla T4
  Outputs: /kaggle/working/E1


## 1. Data: the same 6,443 pairs and the same 645 validation images as every D4 run
Cell 3 is copied verbatim from the D4 notebooks, so the split matches R1–R10 exactly.

In [3]:
# ── Cell 3: Load manifest and build the 4-class masks (verbatim from the D4 notebooks) ──
import cv2
import numpy as np
import os
from collections import Counter, defaultdict
from pathlib import Path
import json

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data):,} entries from manifest")

# ── Filter: ONLY Building Damage Assessment segmentation entries ──
bda_entries = [
    e for e in raw_data
    if e.get("task") == "Referring Expression Segmentation"
    and e.get("cls_description") == "Building Damage Assessment"
]
print(f"Building Damage Assessment entries: {len(bda_entries):,}")

# ── Path resolution (same logic as before) ──
def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "masks" / "masks" / filename,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return None

# ── Group entries by underlying post-disaster image ──
by_image = defaultdict(dict)  # post_image_path -> {folder_type: mask_path}

for e in bda_entries:
    post_img = e.get("post_image_path", "")
    if e.get("image_type") != "Optical":   # skip SAR duplicates, keep optical only
        continue
    mask_rel = e.get("ground_truth", "")
    folder = Path(mask_rel.replace("\\", "/")).parent.name  # e.g. train_building_intact_mask
    resolved = resolve_path(mask_rel)
    if resolved:
        by_image[post_img][folder] = resolved
        by_image[post_img]["post_image_path"] = post_img

print(f"Unique images with at least one mask: {len(by_image):,}")

# ── Build combined (image, mask) pairs ──
FOLDER_TO_CLASS = {
    "train_building_intact_mask": 1,
    "train_building_damaged_mask": 2,
    "train_building_destroyed_mask": 3,   # use 2 here instead if you want to collapse into "damaged"
}

# ── FIX: Create temporary directory on Kaggle hard drive for masks ──
os.makedirs("/tmp/combined_masks", exist_ok=True)

pairs = []
skipped_img = 0

for post_img, mask_dict in by_image.items():
    img_path = resolve_path(post_img)
    if not img_path:
        skipped_img += 1
        continue

    combined_mask = None
    for folder, class_id in FOLDER_TO_CLASS.items():
        mask_path = mask_dict.get(folder)
        if mask_path is None:
            continue
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        if combined_mask is None:
            combined_mask = np.zeros_like(m, dtype=np.uint8)
        combined_mask[m > 0] = class_id   # later classes overwrite earlier if they overlap

    if combined_mask is None:
        continue  # no valid masks found for this image at all

    # ── FIX: Save mask to hard drive instead of RAM ──
    mask_filename = Path(img_path).name
    save_path = f"/tmp/combined_masks/{mask_filename}"
    cv2.imwrite(save_path, combined_mask)

    pairs.append({
        "image_path": img_path,
        "mask_path": save_path,   # Store the string path, NOT the array!
    })

print(f"\n✓ Built {len(pairs):,} valid (image, mask) pairs")
print(f"  Skipped (image not found): {skipped_img:,}")

# Sanity check
if pairs:
    print(f"\n  Sample: {pairs[0]['image_path']}")
    print(f"  Mask saved at: {pairs[0]['mask_path']}")

Loaded 92,968 entries from manifest
Building Damage Assessment entries: 14,531
Unique images with at least one mask: 6,443

✓ Built 6,443 valid (image, mask) pairs
  Skipped (image not found): 0

  Sample: /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct/train_images/train_images/bata_explosion_post_0.png
  Mask saved at: /tmp/combined_masks/bata_explosion_post_0.png


In [4]:
# ── Cell 4: Validation split (identical to every D4 run) and event names ──
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split

NORMALIZE = A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
val_transform = A.Compose([NORMALIZE, ToTensorV2()])   # E-protocol: full native image, no resize

train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)
if SMOKE_TEST:
    val_pairs = val_pairs[:8]

XBD_NAME = re.compile(r"^(.+?)_(\d{8})_post_disaster")
OTHER_NAME = re.compile(r"^(.+?)_post")

def event_and_source(image_path):
    """'hurricane_florence_00000261_post_disaster.png' -> ('hurricane_florence', 'xBD')
    'turkey_earthquake5_post_12.png'                 -> ('turkey_earthquake', 'other')"""
    name = Path(image_path).name
    m = XBD_NAME.match(name)
    if m:
        return m.group(1), "xBD"
    m = OTHER_NAME.match(name)
    event = m.group(1) if m else Path(name).stem
    return re.sub(r"\d+$", "", event), "other"   # merge sub-areas such as turkey_earthquake0..7

events = [event_and_source(p["image_path"]) for p in val_pairs]
n_xbd = sum(1 for _, source in events if source == "xBD")
print(f"✓ Validation images: {len(val_pairs)} ({n_xbd} xBD, {len(val_pairs) - n_xbd} other); "
      f"expected 645 (560 xBD, 85 other) when SMOKE_TEST is False")
print(f"  Events: {len(set(e for e, _ in events))}")

/usr/local/lib/python3.13/dist-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.21). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


✓ Validation images: 645 (560 xBD, 85 other); expected 645 (560 xBD, 85 other) when SMOKE_TEST is False
  Events: 24


## 2. Model, 8-view TTA, metrics and E2 majority voting
The prediction code matches the D4 notebooks (padding to a multiple of 32, the same 8 dihedral views, mixed precision), so the scores reproduce the D4 CSVs.

In [5]:
# ── Cell 5: Model, 8-view TTA, metrics and E2 majority voting ─────────────
import segmentation_models_pytorch as smp
import torch.nn.functional as F

def load_model(path):
    """D4 checkpoints: U-Net + ResNet-50, 4 classes. Accepts a state dict or a resume checkpoint."""
    state = torch.load(path, map_location=DEVICE, weights_only=False)
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    model = smp.Unet(encoder_name="resnet50", encoder_weights=None, in_channels=3, classes=NUM_CLASSES)
    model.load_state_dict(state)
    return model.to(DEVICE).eval()

def _forward_once(model, x):
    """Logits at x's own HxW: pad to a multiple of 32, exactly as in the D4 notebooks."""
    h, w = x.shape[-2:]
    ph, pw = (32 - h % 32) % 32, (32 - w % 32) % 32
    if ph or pw:
        x = F.pad(x, (0, pw, 0, ph), mode="reflect")
    return model(x)[..., :h, :w].float()

# The D4 notebooks' 8 dihedral views: rotation 0/90/180/270 × (no flip, horizontal flip)
TTA_VIEWS = [(k, flip) for flip in (False, True) for k in range(4)]

def _tta_fwd(x, k, flip):
    x = torch.rot90(x, k, dims=(2, 3))
    return torch.flip(x, dims=(3,)) if flip else x

def _tta_inv(y, k, flip):
    if flip:
        y = torch.flip(y, dims=(3,))
    return torch.rot90(y, -k, dims=(2, 3))

@torch.no_grad()
def predict_std_and_tta(model, x):
    """One pass over the 8 views. The identity view (k=0, no flip) is the standard prediction."""
    prob_sum, pred_std = 0, None
    with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
        for k, flip in TTA_VIEWS:
            logits = _forward_once(model, _tta_fwd(x, k, flip))
            if k == 0 and not flip:
                pred_std = logits.argmax(1)
            prob_sum = prob_sum + torch.softmax(_tta_inv(logits, k, flip), dim=1)
    pred_tta = (prob_sum / len(TTA_VIEWS)).argmax(1)
    return (pred_std[0].cpu().numpy().astype(np.uint8),
            pred_tta[0].cpu().numpy().astype(np.uint8))

def confusion(gt, pred):
    """4×4 pixel counts, rows = truth, columns = prediction."""
    idx = gt.astype(np.int64).ravel() * NUM_CLASSES + pred.astype(np.int64).ravel()
    return np.bincount(idx, minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)

def metrics(cm):
    """Dataset-level metrics from a confusion matrix, same formulas as the D4 notebooks,
    plus building IoU (classes 1–3 merged) and where the missed destroyed pixels went."""
    cm = cm.astype(np.float64)
    eps = 1e-8
    tp = np.diag(cm)
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    iou = tp / (tp + fp + fn + eps)
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    building_tp = cm[1:, 1:].sum()
    building_iou = building_tp / (building_tp + cm[0, 1:].sum() + cm[1:, 0].sum() + eps)
    missed = cm[3, :3].sum()
    return {
        "miou": float(iou.mean()), "iou": iou, "precision": precision, "recall": recall,
        "f1": 2 * precision * recall / (precision + recall + eps),
        "f2": 5 * precision * recall / (4 * precision + recall + eps),
        "building_iou": float(building_iou),
        "destroyed_missed_px": int(missed),
        "destroyed_missed_as": {CLASS_NAMES[c]: (float(cm[3, c] / missed) if missed else 0.0)
                                for c in range(3)},
    }

def majority_vote(pred, min_area=0):
    """E2: every predicted building blob (classes 1–3 merged, 8-connected) takes its majority
    class among 1–3. Blobs smaller than min_area keep their pixel labels. Ties go to the lower class."""
    n, labels = cv2.connectedComponents((pred > 0).astype(np.uint8), connectivity=8)
    if n <= 1:
        return pred
    counts = np.bincount(labels.ravel() * NUM_CLASSES + pred.ravel().astype(np.int64),
                         minlength=n * NUM_CLASSES).reshape(n, NUM_CLASSES)
    majority = counts[:, 1:].argmax(1).astype(np.uint8) + 1
    majority[0] = 0                      # label 0 is the background
    voted = majority[labels]
    if min_area:
        keep = counts.sum(1) < min_area  # small blobs keep their own pixel labels
        keep[0] = True
        voted = np.where(keep[labels], pred, voted)
    return voted.astype(np.uint8)

# Self-check: a 4-pixel blob with 3 destroyed pixels and 1 intact pixel becomes all destroyed
_toy = np.zeros((5, 5), np.uint8)
_toy[1, 1:4] = 3
_toy[2, 2] = 1
assert (majority_vote(_toy)[[1, 1, 1, 2], [1, 2, 3, 2]] == 3).all()
assert majority_vote(_toy, min_area=10)[2, 2] == 1     # the blob is below 10 px, so it is unchanged
assert (metrics(confusion(_toy, _toy))["iou"][[0, 1, 3]] > 0.99).all()
print("✓ Helpers ready (majority-vote and metric self-checks passed)")

✓ Helpers ready (majority-vote and metric self-checks passed)


## 3. Run every model over the validation images
For each image this computes the standard and TTA predictions, the four E2 voting variants, and their confusion matrices (overall, per event, and per image). R9's masks and the ground truth are saved as PNGs (values 0–3).

In [6]:
# ── Cell 6: Run every model over the validation images ─────────────────────
from tqdm.auto import tqdm

VARIANTS = ["std", "tta", "std_vote", "tta_vote", f"std_vote{VOTE_MIN_AREA}", f"tta_vote{VOTE_MIN_AREA}"]
cms = {name: {v: np.zeros((NUM_CLASSES, NUM_CLASSES), np.int64) for v in VARIANTS} for name in MODELS}
event_cms = {name: defaultdict(lambda: np.zeros((NUM_CLASSES, NUM_CLASSES), np.int64)) for name in MODELS}
per_image_rows = []

pred_dirs = {v: OUT_DIR / "preds" / f"{EXPORT_PREDS_FOR}_{v}" for v in ("std", "tta")}
gt_dir = OUT_DIR / "gt"
for d in [*pred_dirs.values(), gt_dir]:
    d.mkdir(parents=True, exist_ok=True)

t_start = time.time()
for name, ckpt in MODELS.items():
    model = load_model(ckpt)
    skipped = 0
    for pair, (event, source) in tqdm(list(zip(val_pairs, events)), desc=name):
        image = cv2.imread(pair["image_path"])
        gt = cv2.imread(pair["mask_path"], cv2.IMREAD_GRAYSCALE)
        if image is None or gt is None:
            skipped += 1
            continue
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if gt.shape != image.shape[:2]:   # same guard as the D4 dataset class
            gt = cv2.resize(gt, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)
        x = val_transform(image=image)["image"].unsqueeze(0).to(DEVICE)
        pred_std, pred_tta = predict_std_and_tta(model, x)

        preds = {
            "std": pred_std,
            "tta": pred_tta,
            "std_vote": majority_vote(pred_std),
            "tta_vote": majority_vote(pred_tta),
            f"std_vote{VOTE_MIN_AREA}": majority_vote(pred_std, VOTE_MIN_AREA),
            f"tta_vote{VOTE_MIN_AREA}": majority_vote(pred_tta, VOTE_MIN_AREA),
        }
        row = {"model": name, "image": Path(pair["image_path"]).name, "event": event, "source": source}
        for v, p in preds.items():
            cm = confusion(gt, p)
            cms[name][v] += cm
            if v in ("std", "tta"):
                event_cms[name][(event, source, v)] += cm
                row.update({f"{v}_{t}{q}": int(cm[t, q])
                            for t in range(NUM_CLASSES) for q in range(NUM_CLASSES)})
        per_image_rows.append(row)

        if name == EXPORT_PREDS_FOR:
            fname = Path(pair["image_path"]).stem + ".png"
            cv2.imwrite(str(pred_dirs["std"] / fname), pred_std)
            cv2.imwrite(str(pred_dirs["tta"] / fname), pred_tta)
            cv2.imwrite(str(gt_dir / fname), gt)

    print(f"✓ {name}: {len(val_pairs) - skipped} images scored, {skipped} skipped "
          f"({(time.time() - t_start) / 60:.1f} min so far)")
    del model
    torch.cuda.empty_cache()

R9:   0%|          | 0/645 [00:00<?, ?it/s]

✓ R9: 645 images scored, 0 skipped (6.3 min so far)


R1:   0%|          | 0/645 [00:00<?, ?it/s]

✓ R1: 645 images scored, 0 skipped (12.4 min so far)


R9s7:   0%|          | 0/645 [00:00<?, ?it/s]

✓ R9s7: 645 images scored, 0 skipped (18.5 min so far)


## 4. Results: E1 diagnosis, Gate G2, sanity check and E2 voting
**Gate G2** (Phase2_Plan §3): if 40% or more of the champion's missed destroyed pixels are predicted as **Background**, the model doesn't recognise rubble as a building, so run **R11** (before + after input) first. Otherwise the building is found but graded wrongly, so run **R12** (+1,007 xBD tiles) first.

**E2 adoption rule:** keep voting only if Destroyed IoU rises by more than 0.005 without lowering mIoU, on **both** R9 and R9s7.

In [7]:
# ── Cell 7: E1 results, sanity check, Gate G2 and E2 voting ───────────────
import csv
import pandas as pd

# 1) Overall summary per model and mode, with the sanity check against the D4 CSVs
summary_rows = []
for name in MODELS:
    for v in ("std", "tta"):
        m = metrics(cms[name][v])
        exp_miou, exp_destroyed = EXPECTED[name][v]
        matches = abs(m["miou"] - exp_miou) < 5e-4 and abs(m["iou"][3] - exp_destroyed) < 5e-4
        summary_rows.append({
            "model": name, "mode": v, "miou": round(m["miou"], 4),
            **{f"{c.lower()}_iou": round(float(m["iou"][i]), 4) for i, c in enumerate(CLASS_NAMES)},
            "building_iou": round(m["building_iou"], 4),
            "destroyed_precision": round(float(m["precision"][3]), 4),
            "destroyed_recall": round(float(m["recall"][3]), 4),
            "missed_destroyed_px": m["destroyed_missed_px"],
            "missed_as_background": round(m["destroyed_missed_as"]["Background"], 4),
            "missed_as_intact": round(m["destroyed_missed_as"]["Intact"], 4),
            "missed_as_damaged": round(m["destroyed_missed_as"]["Damaged"], 4),
            "matches_d4_csv": "n/a (smoke test)" if SMOKE_TEST else ("yes" if matches else "NO"),
        })
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT_DIR / "E1_summary.csv", index=False)
print(summary.to_string(index=False))
if not SMOKE_TEST and (summary["matches_d4_csv"] == "NO").any():
    print("\n⚠ A model does not reproduce its D4 CSV: check the checkpoint (wrong run or not the final epoch).")

with open(OUT_DIR / "E1_confusion_matrices.json", "w") as f:
    json.dump({n: {v: cms[n][v].tolist() for v in VARIANTS} for n in MODELS}, f, indent=1)

# 2) Per source (xBD vs other) and per event
print("\nPer source (standard prediction):")
for name in MODELS:
    for src in ("xBD", "other"):
        cm = sum((c for (e, s, v), c in event_cms[name].items() if s == src and v == "std"),
                 np.zeros((NUM_CLASSES, NUM_CLASSES), np.int64))
        m = metrics(cm)
        print(f"  {name:>4} {src:>5}: Destroyed IoU {m['iou'][3]:.4f} | building IoU {m['building_iou']:.4f} | "
              f"missed destroyed -> Background {m['destroyed_missed_as']['Background']:.1%}")

event_rows = []
for name in MODELS:
    for (event, source, v), cm in event_cms[name].items():
        m = metrics(cm)
        event_rows.append({
            "model": name, "mode": v, "event": event, "source": source,
            "destroyed_gt_px": int(cm[3].sum()),
            "destroyed_iou": round(float(m["iou"][3]), 4),
            "destroyed_recall": round(float(m["recall"][3]), 4),
            "building_iou": round(m["building_iou"], 4),
            "missed_as_background": round(m["destroyed_missed_as"]["Background"], 4),
        })
per_event = pd.DataFrame(event_rows).sort_values(["model", "mode", "destroyed_gt_px"],
                                                 ascending=[True, True, False])
per_event.to_csv(OUT_DIR / "E1_per_event.csv", index=False)
pd.DataFrame(per_image_rows).to_csv(OUT_DIR / "E1_per_image_cm.csv", index=False)
print(f"\nPer event, {EXPORT_PREDS_FOR} standard (events with destroyed pixels, most first):")
view = per_event[(per_event["model"] == EXPORT_PREDS_FOR) & (per_event["mode"] == "std")
                 & (per_event["destroyed_gt_px"] > 0)]
print(view[["event", "source", "destroyed_gt_px", "destroyed_iou", "destroyed_recall",
            "missed_as_background"]].to_string(index=False))

# 3) Gate G2, decided on the champion's standard prediction (R1 shown for comparison)
def background_share(name, mode="std"):
    row = summary[(summary["model"] == name) & (summary["mode"] == mode)].iloc[0]
    return float(row["missed_as_background"])

share = background_share(EXPORT_PREDS_FOR)
run_first = "R11 (before + after input)" if share >= 0.40 else "R12 (+1,007 xBD tiles)"
print(f"\nGate G2: {share:.1%} of {EXPORT_PREDS_FOR}'s missed destroyed pixels are predicted as "
      f"Background (R1: {background_share('R1'):.1%}). Rule: 40% or more -> R11 first.")
print(f"  => Run {run_first} first.")

# 4) E2 majority voting, written in the D4 CSV format
CSV_FIELDS = ["member", "experiment_id", "category", "variable", "value", "miou", "bg_iou",
              "intact_iou", "damaged_iou", "destroyed_iou", "precision", "recall", "f1", "f2",
              "destroyed_precision", "destroyed_recall", "destroyed_f2", "train_time_hrs", "seed", "notes"]

def result_row(exp_id, m, value, notes):
    return {
        "member": "Phase 2 (E2)", "experiment_id": exp_id, "category": "E-protocol",
        "variable": "post-processing", "value": value, "miou": round(m["miou"], 4),
        "bg_iou": round(float(m["iou"][0]), 4), "intact_iou": round(float(m["iou"][1]), 4),
        "damaged_iou": round(float(m["iou"][2]), 4), "destroyed_iou": round(float(m["iou"][3]), 4),
        "precision": round(float(m["precision"].mean()), 4), "recall": round(float(m["recall"].mean()), 4),
        "f1": round(float(m["f1"].mean()), 4), "f2": round(float(m["f2"].mean()), 4),
        "destroyed_precision": round(float(m["precision"][3]), 4),
        "destroyed_recall": round(float(m["recall"][3]), 4), "destroyed_f2": round(float(m["f2"][3]), 4),
        "train_time_hrs": None, "seed": None, "notes": notes,
    }

vote_rows = [result_row(f"E2_{name}_{v}", metrics(cms[name][v]), f"{name} {v}",
                        "E-protocol, last epoch; vote = object-based majority voting")
             for name in MODELS for v in VARIANTS]
with open(OUT_DIR / "E2_voting.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
    writer.writeheader()
    writer.writerows(vote_rows)

print("\nE2 majority voting (change against the same model without voting):")
adopt = {}
for name in MODELS:
    for base in ("std", "tta"):
        b = metrics(cms[name][base])
        for v in (f"{base}_vote", f"{base}_vote{VOTE_MIN_AREA}"):
            m = metrics(cms[name][v])
            d_miou, d_destroyed = m["miou"] - b["miou"], m["iou"][3] - b["iou"][3]
            adopt[(name, v)] = d_destroyed > 0.005 and d_miou >= 0
            print(f"  {name:>4} {v:>12}: mIoU {m['miou']:.4f} ({d_miou:+.4f}) | "
                  f"Destroyed {m['iou'][3]:.4f} ({d_destroyed:+.4f})")
both_seeds = [v for v in VARIANTS[2:] if adopt.get(("R9", v)) and adopt.get(("R9s7", v))]
print(f"\nE2 verdict: {'adopt ' + ', '.join(both_seeds) if both_seeds else 'do not adopt voting'} "
      f"(rule: Destroyed IoU +0.005 or more without lowering mIoU, on both R9 and R9s7)")

model mode   miou  background_iou  intact_iou  damaged_iou  destroyed_iou  building_iou  destroyed_precision  destroyed_recall  missed_destroyed_px  missed_as_background  missed_as_intact  missed_as_damaged matches_d4_csv
   R9  std 0.5373          0.9530      0.5571       0.3471         0.2921        0.5840               0.4125            0.5001               903850                0.5950            0.2398             0.1653            yes
   R9  tta 0.5440          0.9541      0.5650       0.3568         0.3001        0.5905               0.4174            0.5163               874559                0.6017            0.2189             0.1794            yes
   R1  std 0.5374          0.9539      0.5693       0.3578         0.2686        0.5929               0.3871            0.4674               962930                0.5941            0.2353             0.1706            yes
   R1  tta 0.5440          0.9543      0.5732       0.3661         0.2822        0.5962               0.4006    

In [8]:
# ── Cell 8: config.json and README for the output folder ──────────────────
def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: fall back to a plain-text table
        return "```\n" + df.to_string(index=False) + "\n```"

config = {
    "task": "E1 + E2", "smoke_test": SMOKE_TEST, "val_images": len(val_pairs),
    "checkpoints": {name: str(path) for name, path in MODELS.items()},   # which file each model loaded
    "export_preds_for": EXPORT_PREDS_FOR, "vote_min_area": VOTE_MIN_AREA, "variants": VARIANTS,
    "gate_g2": {"background_share": round(share, 4), "run_first": run_first},
    "e2_adopted": both_seeds, "run_time_min": round((time.time() - t_start) / 60, 1),
}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")

lines = [
    "# E1 + E2 results",
    "",
    f"Validation images: {len(val_pairs)}{' (SMOKE TEST)' if SMOKE_TEST else ''}. "
    f"Models: {', '.join(MODELS)}. Run time: {(time.time() - t_start) / 60:.0f} min.",
    "",
    "## E1 summary",
    "",
    as_markdown(summary),
    "",
    "## Gate G2",
    "",
    f"{share:.1%} of {EXPORT_PREDS_FOR}'s missed destroyed pixels are predicted as Background "
    f"(rule: 40% or more means R11 first), so run **{run_first}** first.",
    "",
    "## E2 majority voting (D4 CSV format)",
    "",
    as_markdown(pd.DataFrame(vote_rows)[["experiment_id", "miou", "damaged_iou", "destroyed_iou",
                                         "destroyed_recall"]]),
    "",
    f"Verdict: {'adopt ' + ', '.join(both_seeds) if both_seeds else 'do not adopt voting'}.",
    "",
    "## Files",
    "",
    "- `E1_summary.csv`, `E1_per_event.csv`, `E1_per_image_cm.csv`, `E1_confusion_matrices.json`, "
    "`E2_voting.csv`, `config.json`",
    f"- `preds/{EXPORT_PREDS_FOR}_std/`, `preds/{EXPORT_PREDS_FOR}_tta/`, `gt/`: uint8 PNG masks, "
    "0 background, 1 intact, 2 damaged, 3 destroyed (same file names as the post-disaster images)",
]
(OUT_DIR / "README.md").write_text("\n".join(lines), encoding="utf-8")
n_files = sum(1 for _ in OUT_DIR.rglob("*") if _.is_file())
print(f"✓ Wrote config.json and README.md ({n_files} files in {OUT_DIR})")
print("Next: create a Kaggle dataset from this output called d4-r9-val-predictions (used by H1, T1 and H2).")

✓ Wrote config.json and README.md (1942 files in /kaggle/working/E1)
Next: create a Kaggle dataset from this output called d4-r9-val-predictions (used by H1, T1 and H2).
